In [1]:
import sys
import os
import subprocess
from pathlib import Path

# Ensure project import path
PROJECT_ROOT = Path('/global/home/hpc5656/SLAM')
sys.path.append(str(PROJECT_ROOT))
# Set working directory so relative paths (e.g., src/config/*.yaml) resolve
os.chdir(str(PROJECT_ROOT))
print('CWD:', Path.cwd())

%matplotlib inline

# CuPy is required - ensure CUDA_PATH and LD_LIBRARY_PATH are set
# This allows the notebook to work even if Jupyter wasn't started with modules loaded
if "CUDA_PATH" not in os.environ:
    print("CUDA_PATH not set, attempting to load modules...")
    try:
        result = subprocess.run(
            'module load cuda/12.2 && env',
            shell=True,
            executable='/bin/bash',
            capture_output=True,
            text=True,
            timeout=10
        )
        if result.returncode == 0:
            for line in result.stdout.split('\n'):
                if '=' in line:
                    key, value = line.split('=', 1)
                    os.environ[key] = value
            print(f"✓ Modules loaded. CUDA_PATH: {os.environ.get('CUDA_PATH', 'Not set')}")
        else:
            print(f"⚠ Could not load modules. Error: {result.stderr}")
            raise RuntimeError(
                "CUDA_PATH not set and could not load modules. "
                "Please run: module load cuda/12.2 before starting Jupyter."
            )
    except Exception as e:
        print(f"✗ Could not load modules: {e}")
        raise RuntimeError(
            f"Failed to load CUDA modules: {e}\n"
            "Please ensure CUDA is loaded before starting Jupyter:\n"
            "  module load cuda/12.2"
        ) from e

# Ensure LD_LIBRARY_PATH includes CUDA library directory for NVRTC (libnvrtc.so.12)
cuda_path = os.environ.get('CUDA_PATH')
libnvrtc_path = None

if cuda_path:
    cuda_lib_paths = [
        os.path.join(cuda_path, 'lib64'),
        os.path.join(cuda_path, 'targets', 'x86_64-linux', 'lib'),
    ]
    
    current_ld_path = os.environ.get('LD_LIBRARY_PATH', '')
    ld_paths = current_ld_path.split(':') if current_ld_path else []
    paths_added = []
    
    for cuda_lib_path in cuda_lib_paths:
        if os.path.exists(cuda_lib_path):
            if cuda_lib_path not in ld_paths:
                paths_added.append(cuda_lib_path)
                ld_paths.insert(0, cuda_lib_path)
    
    if paths_added:
        os.environ['LD_LIBRARY_PATH'] = ':'.join(ld_paths)
        print(f"✓ Updated LD_LIBRARY_PATH to include: {', '.join(paths_added)}")
    
    for cuda_lib_path in cuda_lib_paths:
        if os.path.exists(cuda_lib_path):
            potential_libnvrtc = os.path.join(cuda_lib_path, 'libnvrtc.so.12')
            if os.path.exists(potential_libnvrtc):
                libnvrtc_path = potential_libnvrtc
                print(f"✓ Found libnvrtc.so.12 at: {libnvrtc_path}")
                try:
                    import ctypes
                    try:
                        lib = ctypes.CDLL(libnvrtc_path, mode=ctypes.RTLD_GLOBAL)
                        print(f"✓ Preloaded libnvrtc.so.12 using ctypes (RTLD_GLOBAL)")
                    except Exception as e1:
                        try:
                            lib = ctypes.CDLL(libnvrtc_path)
                            print(f"✓ Preloaded libnvrtc.so.12 using ctypes (standard)")
                        except Exception as e2:
                            raise e1 from e2
                except Exception as e:
                    print(f"⚠ Warning: Could not preload libnvrtc.so.12: {e}")
                
                try:
                    import ctypes.util
                    found_lib = ctypes.util.find_library('nvrtc')
                    if found_lib:
                        print(f"✓ ctypes.util.find_library('nvrtc') found: {found_lib}")
                except Exception:
                    pass
                break

from src.utils.array_backend import np, random, is_cupy
from src.classes.belief_mdp_n_M import BeliefMDP_n_M_SLAM
from src.classes.model import DoubleIntegratorModel, LIDAR
from src.classes.mapping import LidarGridMapVec
from src.utils.map import load_obstacles_config
from tqdm import tqdm
import time
import warnings

# Suppress CuPy experimental FutureWarnings
warnings.filterwarnings('ignore', category=FutureWarning, module='cupy.random')

print(f"✓ All imports successful")
print(f"Using backend: {'CuPy (GPU)' if is_cupy else 'NumPy (CPU)'}")

# Verify we're using CuPy
if not is_cupy:
    raise RuntimeError(
        "CuPy is required but not being used. "
        "Check CUDA installation and CuPy setup."
    )

"""
Generate p_n^{(M)} transition probability matrix for BeliefMDP_n_M_SLAM.

UPDATED: This notebook has been updated to reflect changes in pomdp.py and belief_mdp_n.py.
The current implementation uses discrete observation quantization (Y_n) instead of Monte Carlo integration.

Key changes:
- η_n_batch is now fully vectorized (no for loops)
- Uses discrete observation quantization via Q_n matrix (no MC integration)
- Significantly faster computation using GPU-accelerated vectorized operations

This notebook:
1. Creates a BeliefMDP_n_M_SLAM instance with coarse quantization (n=2)
2. Automatically computes and caches p_n^{(M)} for all belief-action pairs
3. Verifies the computed matrix properties
4. Provides statistics and validation

Uses the same model setup:
- DoubleIntegratorModel with n=2, dt=1.0, max_a=2.0
- LIDAR(fov=360, r_max=10.0, B=8)
- M parameter controls belief space quantization
- Observation quantization uses obs_n (defaults to n)
"""


CWD: /global/home/hpc5656/SLAM
CUDA_PATH not set, attempting to load modules...
✓ Modules loaded. CUDA_PATH: /cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v3/Core/cudacore/12.2.2
✓ Updated LD_LIBRARY_PATH to include: /cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v3/Core/cudacore/12.2.2/lib64, /cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v3/Core/cudacore/12.2.2/targets/x86_64-linux/lib
✓ Found libnvrtc.so.12 at: /cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v3/Core/cudacore/12.2.2/lib64/libnvrtc.so.12
✓ Preloaded libnvrtc.so.12 using ctypes (RTLD_GLOBAL)
✓ ctypes.util.find_library('nvrtc') found: libnvrtc.so.12
✓ Using CuPy for GPU acceleration
✓ Using cupyx.scipy.spatial.KDTree
✓ All imports successful
Using backend: CuPy (GPU)


'\nGenerate p_n^{(M)} transition probability matrix for BeliefMDP_n_M_SLAM.\n\nUPDATED: This notebook has been updated to reflect changes in pomdp.py and belief_mdp_n.py.\nThe current implementation uses discrete observation quantization (Y_n) instead of Monte Carlo integration.\n\nKey changes:\n- η_n_batch is now fully vectorized (no for loops)\n- Uses discrete observation quantization via Q_n matrix (no MC integration)\n- Significantly faster computation using GPU-accelerated vectorized operations\n\nThis notebook:\n1. Creates a BeliefMDP_n_M_SLAM instance with coarse quantization (n=2)\n2. Automatically computes and caches p_n^{(M)} for all belief-action pairs\n3. Verifies the computed matrix properties\n4. Provides statistics and validation\n\nUses the same model setup:\n- DoubleIntegratorModel with n=2, dt=1.0, max_a=2.0\n- LIDAR(fov=360, r_max=10.0, B=8)\n- M parameter controls belief space quantization\n- Observation quantization uses obs_n (defaults to n)\n'

In [2]:
# Configuration
n = 2  # Coarse quantization for initial testing
M = 2  # Belief space quantization parameter (controls cardinality)
beta = 0.95  # Discount factor

print(f"Configuration:")
print(f"  n (state quantization): {n}")
print(f"  M (belief quantization): {M}")
print(f"  β (discount factor): {beta}")
print(f"\nThis will create a BeliefMDP_n_M instance which will:")
print(f"  1. Load or generate belief codebook (Π_n^M)")
print(f"  2. Load cached T_mat")
print(f"  3. Compute p_n^{(M)} for all belief-action pairs")
print(f"  4. Cache the result for future use")

# Load environment configuration
obstacles, area = load_obstacles_config(environment='toy2')

# Create models
motion_model = DoubleIntegratorModel(p_x=5.0, p_y=5.0, v_x=0.0, v_y=0.0, dt=1.0, max_a=2.0)
sensor = LIDAR(fov=360, r_max=10.0, B=5)
grid_map = LidarGridMapVec(
    x_min=area[0], x_max=area[1],
    y_min=area[2], y_max=area[3],
    quantization_level=n,
)

print(f"Environment setup:")
print(f"  Map bounds: x=[{area[0]}, {area[1]}], y=[{area[2]}, {area[3]}]")
print(f"  Map quantization: {n}x{n} = {n**2} cells")
print(f"  Total maps: 2^{n**2} = {2**(n**2)}")
print(f"  Motion model: DoubleIntegratorModel (dt={motion_model.dt}, max_a={motion_model.max_a})")
print(f"  Sensor: LIDAR (fov={sensor.fov}°, r_max={sensor.r_max}, B={sensor.B})")

Configuration:
  n (state quantization): 2
  M (belief quantization): 2
  β (discount factor): 0.95

This will create a BeliefMDP_n_M instance which will:
  1. Load or generate belief codebook (Π_n^M)
  2. Load cached T_mat
  3. Compute p_n^2 for all belief-action pairs
  4. Cache the result for future use
Environment setup:
  Map bounds: x=[0, 10], y=[0, 10]
  Map quantization: 2x2 = 4 cells
  Total maps: 2^4 = 16
  Motion model: DoubleIntegratorModel (dt=1.0, max_a=2.0)
  Sensor: LIDAR (fov=6.283185307179586°, r_max=10.0, B=5)


In [3]:
# Create BeliefMDP_n_M_SLAM instance
# This will automatically:
# 1. Load cached T_mat (or compute if not found)
# 2. Load cached Q_n (observation quantization) or compute if not found
# 3. Load cached belief codebook (or generate if not found)
# 4. Compute p_n^{(M)} for all actions using vectorized η_n_batch (or load from cache if exists)

print("="*70)
print("Creating BeliefMDP_n_M_SLAM instance...")
print("="*70)
print()

start_time = time.time()

bmdp_M = BeliefMDP_n_M_SLAM(
    M=M,
    β=beta,
    n=n,
    motion_model=motion_model,
    measurement_model=sensor,
    obstacles=obstacles,
    _map=grid_map,
    sigma_v=1.0
)

# Seed map from obstacles
bmdp_M.map.seed_from_obstacles(obstacles)

elapsed_time = time.time() - start_time

print()
print("="*70)
print("Initialization complete!")
print("="*70)
print(f"Total time: {elapsed_time:.2f}s")
print()
print(f"Belief space statistics:")
print(f"  Cardinality |Π_n^M|: {bmdp_M.BQ.cardinality:,}")
print(f"  Belief dimension N_n: {bmdp_M.SQ.m_n * bmdp_M.len_M}")
print(f"  State space size m_n: {bmdp_M.SQ.m_n}")
print(f"  Action space size n_u: {bmdp_M.AQ.n_u}")

# Check if sparse format
is_sparse = isinstance(bmdp_M.p_n_M, list)
if is_sparse:
    print(f"  p_n^{(M)} format: Sparse (CSR, float16)")
    print(f"    - {len(bmdp_M.p_n_M)} actions")
    print(f"    - Each matrix: {bmdp_M.p_n_M[0].shape[0]:,} × {bmdp_M.p_n_M[0].shape[1]:,}")
    total_nnz = sum(mat.nnz for mat in bmdp_M.p_n_M)
    total_entries = bmdp_M.BQ.cardinality * bmdp_M.BQ.cardinality * bmdp_M.AQ.n_u
    sparsity = (1.0 - total_nnz / total_entries) * 100
    print(f"    - Total non-zeros: {total_nnz:,} / {total_entries:,} ({sparsity:.2f}% sparse)")
else:
    print(f"  p_n^{(M)} shape: {bmdp_M.p_n_M.shape}")
    print(f"    - {bmdp_M.p_n_M.shape[0]} belief states (from)")
    print(f"    - {bmdp_M.p_n_M.shape[1]} belief states (to)")
    print(f"    - {bmdp_M.p_n_M.shape[2]} actions")


Creating BeliefMDP_n_M_SLAM instance...

Loaded cached T_mat from /global/home/hpc5656/SLAM/cache/T_mat/T_mat_n2_map2x2_max2.0_ea380e8a.npz
  Checking cache file: Q_n_n2_obs2_map2x2_B5_2bb95e8a.npz
  ✓ Cache metadata matches, loading Q_n
Loaded cached Q_n from /global/home/hpc5656/SLAM/cache/Q_n/Q_n_n2_obs2_map2x2_B5_2bb95e8a.npz
  ✓ Cache validation passed
  ✓ Loaded codebook from cache: cache/belief_quantizer/belief_quantizer_M2_N256.npz
Computing p_n_M for the first time...
This may take a while: 4 actions × 32896² belief transitions
  Auto-selected j_batch_size=32896 (processing all targets at once)
Computing p_n_M for action 1/4...
  This will compute transitions for 32,896 belief states
  Processing 32,896 target beliefs per belief state
  Using i_batch_size=9 (processing 9 source beliefs simultaneously)


Action 1/4:   0%|          | 0/32896 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
def get_gpu_memory_info():
    """
    Get GPU memory information using nvidia-smi and CuPy memory pool.
    
    Returns:
        dict with memory stats: total, used, free, cupy_used, cupy_free
    """
    import subprocess
    
    stats = {}
    
    # Get GPU memory from nvidia-smi
    try:
        result = subprocess.run(
            ['nvidia-smi', '--query-gpu=memory.total,memory.used,memory.free', 
             '--format=csv,nounits,noheader'],
            capture_output=True,
            text=True,
            timeout=5
        )
        if result.returncode == 0:
            lines = result.stdout.strip().split('\n')
            if lines:
                # Get first GPU (index 0)
                values = [int(x.strip()) for x in lines[0].split(',')]
                stats['total_mb'] = values[0]
                stats['used_mb'] = values[1]
                stats['free_mb'] = values[2]
    except Exception as e:
        print(f"⚠ Could not query nvidia-smi: {e}")
    
    # Get CuPy memory pool stats
    if is_cupy:
        try:
            import cupy as cp
            mempool = cp.get_default_memory_pool()
            stats['cupy_used_mb'] = mempool.used_bytes() / (1024**2)
            stats['cupy_free_mb'] = mempool.free_bytes() / (1024**2)
            stats['cupy_total_mb'] = stats['cupy_used_mb'] + stats['cupy_free_mb']
        except Exception as e:
            print(f"⚠ Could not query CuPy memory pool: {e}")
    
    return stats


def test_optimal_j_batch_size(M, n, motion_model, sensor, obstacles, grid_map, 
                              test_j_batch_sizes=None, test_duration_sec=30):
    """
    Test different j_batch_size values by running a small p_n_M computation and monitoring GPU.
    
    This function:
    1. Creates a BeliefMDP_n_M instance (which will start computing p_n_M if not cached)
    2. Monitors GPU usage during computation
    3. Cancels after test_duration_sec to test the next batch size
    4. Finds optimal j_batch_size that maximizes GPU usage without OOM
    
    Args:
        M: Belief quantization parameter
        n: State quantization parameter  
        motion_model: Motion model instance
        sensor: Sensor model instance
        obstacles: Obstacles list
        grid_map: Grid map instance
        test_j_batch_sizes: List of j_batch_size values to test (default: [10, 25, 50, 100, 200, 500, 1000])
        test_duration_sec: How long to run each test before canceling (default: 30 seconds)
    
    Returns:
        dict: Results with optimal batch size and GPU usage metrics
    """
    import signal
    import threading
    from src.classes.belief_mdp_n_M import BeliefMDP_n_M_SLAM
    
    if test_j_batch_sizes is None:
        test_j_batch_sizes = [10, 25, 50, 100, 200, 500, 1000, 2000]
    
    # We'll create bmdp_M inside the test, but first let's get expected cardinality
    # by creating a temporary quantizer
    from src.classes.quantizer import BeliefQuantizer
    temp_bq = BeliefQuantizer(M, n * n * 2**(n*n))  # Approximate N_n
    expected_cardinality = temp_bq.cardinality
    del temp_bq
    
    print(f"\n{'='*70}")
    print(f"=== Testing Optimal j_batch_size by Monitoring p_n_M Computation ===")
    print(f"{'='*70}")
    print(f"Expected cardinality: {expected_cardinality:,}")
    print(f"Test duration per batch size: {test_duration_sec} seconds")
    print(f"j_batch_sizes to test: {test_j_batch_sizes}")
    print(f"{'='*70}\n")
    print(f"⚠ This will start p_n_M computation and cancel it after {test_duration_sec}s")
    print(f"⚠ Make sure p_n_M is NOT cached, or delete the cache first!\n")
    
    # Get initial GPU memory state
    initial_mem = get_gpu_memory_info()
    if 'total_mb' in initial_mem:
        print(f"Initial GPU Memory:")
        print(f"  Total: {initial_mem['total_mb']:.0f} MB")
        print(f"  Used: {initial_mem['used_mb']:.0f} MB")
        print(f"  Free: {initial_mem['free_mb']:.0f} MB")
    if 'cupy_used_mb' in initial_mem:
        print(f"  CuPy Used: {initial_mem['cupy_used_mb']:.2f} MB")
        print(f"  CuPy Free: {initial_mem['cupy_free_mb']:.2f} MB")
    print()
    
    results = []
    oom_occurred = False
    
    # Flag to signal cancellation
    cancel_flag = threading.Event()
    max_gpu_usage = {}  # Track max GPU usage per batch size
    
    def monitor_gpu_usage(j_batch_size, duration):
        """Monitor GPU usage during computation"""
        import time as time_module
        start_time = time_module.time()
        gpu_readings = []
        
        while not cancel_flag.is_set() and (time_module.time() - start_time) < duration:
            mem = get_gpu_memory_info()
            if 'used_mb' in mem:
                gpu_readings.append({
                    'time': time_module.time() - start_time,
                    'used_mb': mem['used_mb'],
                    'free_mb': mem.get('free_mb', 0),
                    'cupy_used_mb': mem.get('cupy_used_mb', 0)
                })
            time_module.sleep(0.5)  # Sample every 0.5 seconds
        
        if gpu_readings:
            max_gpu_usage[j_batch_size] = {
                'max_used_mb': max(r['used_mb'] for r in gpu_readings),
                'max_cupy_used_mb': max(r.get('cupy_used_mb', 0) for r in gpu_readings),
                'avg_used_mb': sum(r['used_mb'] for r in gpu_readings) / len(gpu_readings),
                'readings': gpu_readings
            }
    
    for j_batch_size in test_j_batch_sizes:
        print(f"\n{'='*70}")
        print(f"Testing j_batch_size={j_batch_size}")
        print(f"{'='*70}")
        
        # Clear GPU memory before test
        if is_cupy:
            import cupy as cp
            cp.get_default_memory_pool().free_all_blocks()
            cp.get_default_pinned_memory_pool().free_all_blocks()
        
        cancel_flag.clear()
        mem_before = get_gpu_memory_info()
        
        # Start GPU monitoring in background
        monitor_thread = threading.Thread(
            target=monitor_gpu_usage,
            args=(j_batch_size, test_duration_sec + 5),
            daemon=True
        )
        monitor_thread.start()
        
        try:
             # Create BeliefMDP_n_M instance - this will start computing p_n_M
             # Set test j_batch_size as attribute before creation
             print(f"  Creating BeliefMDP_n_M instance...")
             print(f"  (This will start p_n_M computation with j_batch_size={j_batch_size})")
             
             # Create instance - it will try to load cache first
             # We'll set the test j_batch_size after creation but before computation starts
             # Actually, we need to set it before __init__ calls _compute_p_n_M
             # So we'll create it, then if cache doesn't exist, we can modify the method
             
             # For testing, we'll test η_n_batch directly instead of full p_n_M
             # This is faster and doesn't require cache deletion
             print(f"  Testing η_n_batch directly (faster than full p_n_M computation)...")
             
                                     # Set class-level flag to skip p_n_M computation BEFORE creating instance
            BeliefMDP_n_M_SLAM._class_skip_p_n_M_computation = True
            
            # Create instance - p_n_M computation will be skipped
            bmdp_M_test = BeliefMDP_n_M_SLAM(
                M=M,
                β=0.95,
                n=n,
                motion_model=motion_model,
                measurement_model=sensor,
                obstacles=obstacles,
                _map=grid_map,
                sigma_v=1.0
            )
            
            # Reset flag after creation
            BeliefMDP_n_M_SLAM._class_skip_p_n_M_computation = False
             
             # Test η_n_batch with this batch size (now fully vectorized, no MC integration)
             test_i = 0
             test_u = bmdp_M_test.AQ.U[0]
             test_j_list = list(range(min(j_batch_size, bmdp_M_test.BQ.cardinality)))
             
             print(f"  Running vectorized η_n_batch with {len(test_j_list)} target beliefs...")
             start_time = time.time()
             
             probs = bmdp_M_test.η_n_batch(
                 test_j_list, test_i, test_u,
                 show_progress=False
             )
             
             elapsed_time = time.time() - start_time
             
             # Wait a bit more for monitoring to capture peak usage
             time.sleep(1)
             cancel_flag.set()
             monitor_thread.join(timeout=1)
             
             mem_after = get_gpu_memory_info()
             gpu_stats = max_gpu_usage.get(j_batch_size, {})
             
             # Verify results
             if hasattr(probs, 'get'):
                 probs_cpu = probs.get()
             else:
                 probs_cpu = probs
             
             valid_probs = np.sum((probs_cpu >= 0) & (probs_cpu <= 1))
             
             results.append({
                 'j_batch_size': j_batch_size,
                 'time': elapsed_time,
                 'time_per_j': elapsed_time / len(test_j_list) if len(test_j_list) > 0 else 0,
                 'j_per_second': len(test_j_list) / elapsed_time if elapsed_time > 0 else 0,
                 'memory_delta_mb': mem_after.get('cupy_used_mb', 0) - mem_before.get('cupy_used_mb', 0),
                 'memory_used_mb': mem_after.get('cupy_used_mb', 0),
                 'memory_free_mb': mem_after.get('cupy_free_mb', 0),
                 'max_memory_used_mb': gpu_stats.get('max_used_mb', 0),
                 'max_cupy_used_mb': gpu_stats.get('max_cupy_used_mb', 0),
                 'avg_memory_used_mb': gpu_stats.get('avg_used_mb', 0),
                 'valid_probs': int(valid_probs),
                 'total_probs': len(probs_cpu),
                 'success': True,
                 'method': 'direct_eta_n_batch'
             })
             
             print(f"  ✓ Test completed")
             print(f"    Time: {elapsed_time:.4f}s")
             print(f"    Time per j: {elapsed_time/len(test_j_list)*1000:.2f} ms" if len(test_j_list) > 0 else "    Time per j: N/A")
             print(f"    Throughput: {len(test_j_list)/elapsed_time:.1f} j/sec" if elapsed_time > 0 else "    Throughput: N/A")
             if gpu_stats:
                 print(f"    Max GPU memory: {gpu_stats.get('max_used_mb', 0):.0f} MB")
                 print(f"    Max CuPy memory: {gpu_stats.get('max_cupy_used_mb', 0):.2f} MB")
             print(f"    Valid probabilities: {valid_probs}/{len(probs_cpu)}")
             
             # Clean up the test instance
             del bmdp_M_test
             if is_cupy:
                 import cupy as cp
                 cp.get_default_memory_pool().free_all_blocks()
                 cp.get_default_pinned_memory_pool().free_all_blocks()
            
            
                
        except KeyboardInterrupt:
            print(f"\n  ⚠ Interrupted by user")
            cancel_flag.set()
            monitor_thread.join(timeout=1)
            
            mem_after = get_gpu_memory_info()
            gpu_stats = max_gpu_usage.get(j_batch_size, {})
            
            results.append({
                'j_batch_size': j_batch_size,
                'interrupted': True,
                'max_memory_used_mb': gpu_stats.get('max_used_mb', 0),
                'max_cupy_used_mb': gpu_stats.get('max_cupy_used_mb', 0),
                'success': True,
                'method': 'p_n_M_computation_interrupted'
            })
            
        except Exception as e:
            error_msg = str(e)
            cancel_flag.set()
            oom_occurred = True
            
            mem_after = get_gpu_memory_info()
            gpu_stats = max_gpu_usage.get(j_batch_size, {})
            
            results.append({
                'j_batch_size': j_batch_size,
                'success': False,
                'error': error_msg,
                'max_memory_used_mb': gpu_stats.get('max_used_mb', 0),
                'max_cupy_used_mb': gpu_stats.get('max_cupy_used_mb', 0)
            })
            
            print(f"  ✗ Failed: {error_msg}")
            
            if "out of memory" in error_msg.lower() or "OOM" in error_msg:
                print(f"  ⚠ Out of memory - this batch size is too large")
                print(f"  Stopping batch size tests")
                break
            elif "IllegalAddress" in error_msg:
                print(f"  ⚠ GPU memory access error")
        
        # Clean up GPU memory
        if is_cupy:
            import cupy as cp
            cp.get_default_memory_pool().free_all_blocks()
            cp.get_default_pinned_memory_pool().free_all_blocks()
        
        # Small delay between tests
        time.sleep(2)
    
    # Analyze results
    if results:
        successful_results = [r for r in results if r.get('success', False)]
        
        if successful_results:
            print(f"\n{'='*70}")
            print(f"=== Batch Size Analysis ===")
            print(f"{'='*70}")
            print(f"{'j_batch_size':<15s} {'Time (s)':<15s} {'Time/j (ms)':<15s} {'j/sec':<15s} {'Memory (MB)':<15s}")
            print(f"{'-'*70}")
            
            baseline = successful_results[0]
            
            for result in successful_results:
                mem_str = f"{result.get('memory_delta_mb', 0):.1f} Δ"
                print(f"{result['j_batch_size']:<15d} {result['time']:<15.4f} "
                      f"{result['time_per_j']*1000:<15.2f} {result['j_per_second']:<15.1f} {mem_str:<15s}")
            
            # Find optimal: balance between throughput and memory
            # Prefer larger batches with good throughput
            fastest = max(successful_results, key=lambda x: x['j_per_second'])
            
            # Optimal = largest batch with throughput within 10% of fastest
            optimal_candidates = [
                r for r in successful_results
                if r['j_per_second'] >= fastest['j_per_second'] * 0.9
            ]
            
            if optimal_candidates:
                optimal = max(optimal_candidates, key=lambda x: x['j_batch_size'])
                optimal_j_batch_size = optimal['j_batch_size']
                
                print(f"\n{'='*70}")
                print(f"=== Recommendations ===")
                print(f"{'='*70}")
                print(f"✓ Optimal j_batch_size: {optimal_j_batch_size}")
                print(f"  - Throughput: {optimal['j_per_second']:.1f} j/sec")
                print(f"  - Time per j: {optimal['time_per_j']*1000:.2f} ms")
                if optimal.get('memory_delta_mb', 0) > 0:
                    print(f"  - Memory usage: {optimal['memory_delta_mb']:.2f} MB")
                
                if not oom_occurred and optimal_j_batch_size == successful_results[-1]['j_batch_size']:
                    print(f"  ⚠ Consider testing even larger batch sizes (memory allows)")
                elif oom_occurred:
                    print(f"  ⚠ Larger batch sizes cause OOM - this is near the limit")
            else:
                optimal_j_batch_size = fastest['j_batch_size']
                print(f"\n✓ Fastest j_batch_size: {optimal_j_batch_size}")
        else:
            print(f"\n⚠ No successful batch size tests - using default j_batch_size=100")
            optimal_j_batch_size = 100
    else:
        optimal_j_batch_size = 100
    
    return {
        'optimal_j_batch_size': optimal_j_batch_size,
        'results': results
    }


# Run test (will be executed after bmdp_M is created)
print("Ready to test optimal j_batch_size")
print("Run: optimal_batch_results = test_optimal_j_batch_size(bmdp_M)")


In [ ]:
# Test optimal j_batch_size for η_n_batch
# This tests η_n_batch directly (doesn't require p_n_M to be computed)
# and monitors GPU usage to find optimal batch size

print("="*70)
print("Testing optimal j_batch_size for η_n_batch...")
print("="*70)
print()
print("⚠ This will test η_n_batch with different batch sizes")
print("⚠ Each test monitors GPU usage to find optimal settings")
print()

# Run the test (pass parameters needed to create BeliefMDP_n_M)
optimal_batch_results = test_optimal_j_batch_size(
    M=M,
    n=n,
    motion_model=motion_model,
    sensor=sensor,
    obstacles=obstacles,
    grid_map=grid_map,
    test_j_batch_sizes=[10, 25, 50, 100, 200, 500, 1000, 2000],
    test_duration_sec=10  # Short duration since we're testing η_n_batch directly
)

print()
print("="*70)
print("Test complete!")
print("="*70)
if optimal_batch_results:
    print(f"Recommended j_batch_size: {optimal_batch_results.get('optimal_j_batch_size', 100)}")
    print()
    print("You can now use this value when calling _compute_p_n_M:")
    print(f"  bmdp_M._compute_p_n_M(j_batch_size={optimal_batch_results.get('optimal_j_batch_size', 100)})")
else:
    print("⚠ No results available - check test output above")


In [ ]:
# Verify p_n^{(M)} properties
print("="*70)
print("Verifying p_n^{(M)} properties...")
print("="*70)
print()

cardinality = bmdp_M.BQ.cardinality
n_u = bmdp_M.AQ.n_u

# Check if sparse format
is_sparse = isinstance(bmdp_M.p_n_M, list)

if is_sparse:
    print("✓ Using sparse matrix format (CSR, float16)")
    print()
    
    # Check 1: All values are non-negative
    all_min_vals = []
    all_max_vals = []
    all_row_sums = []
    total_nnz = 0
    
    for k in range(n_u):
        p_n_M_k = bmdp_M.p_n_M[k]
        total_nnz += p_n_M_k.nnz
        
        # Get non-zero values
        if p_n_M_k.nnz > 0:
            data = p_n_M_k.data
            all_min_vals.append(float(np.min(data)))
            all_max_vals.append(float(np.max(data)))
            
            # Row sums (sparse matrix row sum)
            row_sums_k = np.array(p_n_M_k.sum(axis=1)).flatten()
            all_row_sums.extend(row_sums_k.tolist())
        else:
            all_row_sums.extend([0.0] * cardinality)
    
    min_val = min(all_min_vals) if all_min_vals else 0.0
    max_val = max(all_max_vals) if all_max_vals else 0.0
    
    print(f"Value range:")
    print(f"  Min: {min_val:.6e}")
    print(f"  Max: {max_val:.6e}")
    if min_val < 0:
        print(f"  ⚠ WARNING: Found negative values!")
    else:
        print(f"  ✓ All values are non-negative")
    
    if max_val > 1.0:
        print(f"  ⚠ WARNING: Found values > 1.0!")
    else:
        print(f"  ✓ All values are ≤ 1.0")
    
    # Check 2: Rows sum to 1 (probability measure)
    print(f"\nRow normalization (should sum to 1.0 for each (belief, action)):")
    all_row_sums_arr = np.array(all_row_sums)
    min_row_sum = float(np.min(all_row_sums_arr))
    max_row_sum = float(np.max(all_row_sums_arr))
    mean_row_sum = float(np.mean(all_row_sums_arr))
    
    print(f"  Min row sum: {min_row_sum:.6e}")
    print(f"  Max row sum: {max_row_sum:.6e}")
    print(f"  Mean row sum: {mean_row_sum:.6e}")
    
    # Check normalization (allow some tolerance for float16 precision)
    normalized_count = np.sum(np.abs(all_row_sums_arr - 1.0) < 1e-3)
    print(f"  Normalized rows: {normalized_count}/{len(all_row_sums_arr)}")
    
    if np.allclose(all_row_sums_arr, 1.0, atol=1e-3):
        print(f"  ✓ All rows sum to 1.0 (within tolerance)")
    else:
        print(f"  ⚠ WARNING: Some rows do not sum to 1.0")
        off_count = np.sum(np.abs(all_row_sums_arr - 1.0) > 1e-3)
        print(f"    {off_count}/{len(all_row_sums_arr)} rows are not normalized")
    
    # Check 3: Sparsity
    total_entries = cardinality * cardinality * n_u
    sparsity = (1.0 - total_nnz / total_entries) * 100
    
    print(f"\nSparsity:")
    print(f"  Non-zero entries: {total_nnz:,} / {total_entries:,}")
    print(f"  Sparsity: {sparsity:.2f}%")
    print(f"  Memory savings: ~{sparsity:.1f}% reduction vs dense")
    
    # Check 4: Sample some transition probabilities
    print(f"\nSample transitions:")
    p_n_M_0 = bmdp_M.p_n_M[0]
    prob_00 = float(p_n_M_0[0, 0]) if p_n_M_0[0, 0] != 0 else 0.0
    prob_01 = float(p_n_M_0[0, 1]) if p_n_M_0[0, 1] != 0 else 0.0
    print(f"  p_n^{(M)}(π_0 → π_0 | u_0): {prob_00:.6e}")
    print(f"  p_n^{(M)}(π_0 → π_1 | u_0): {prob_01:.6e}")
    if cardinality > 2:
        prob_10 = float(p_n_M_0[1, 0]) if p_n_M_0[1, 0] != 0 else 0.0
        prob_11 = float(p_n_M_0[1, 1]) if p_n_M_0[1, 1] != 0 else 0.0
        print(f"  p_n^{(M)}(π_1 → π_0 | u_0): {prob_10:.6e}")
        print(f"  p_n^{(M)}(π_1 → π_1 | u_0): {prob_11:.6e}")
else:
    # Dense format (backward compatibility)
    # Check 1: All values are non-negative
    min_val = float(np.min(bmdp_M.p_n_M))
    max_val = float(np.max(bmdp_M.p_n_M))
    print(f"Value range:")
    print(f"  Min: {min_val:.6e}")
    print(f"  Max: {max_val:.6e}")
    if min_val < 0:
        print(f"  ⚠ WARNING: Found negative values!")
    else:
        print(f"  ✓ All values are non-negative")
    
    if max_val > 1.0:
        print(f"  ⚠ WARNING: Found values > 1.0!")
    else:
        print(f"  ✓ All values are ≤ 1.0")
    
    # Check 2: Rows sum to 1 (probability measure)
    print(f"\nRow normalization (should sum to 1.0 for each (belief, action)):")
    row_sums = bmdp_M.p_n_M.sum(axis=1)  # Sum over target beliefs: (cardinality, n_u)
    min_row_sum = float(np.min(row_sums))
    max_row_sum = float(np.max(row_sums))
    mean_row_sum = float(np.mean(row_sums))
    
    print(f"  Min row sum: {min_row_sum:.6e}")
    print(f"  Max row sum: {max_row_sum:.6e}")
    print(f"  Mean row sum: {mean_row_sum:.6e}")
    
    if np.allclose(row_sums, 1.0, atol=1e-5):
        print(f"  ✓ All rows sum to 1.0 (within tolerance)")
    else:
        print(f"  ⚠ WARNING: Some rows do not sum to 1.0")
        off_count = np.sum(np.abs(row_sums - 1.0) > 1e-5)
        print(f"    {off_count}/{row_sums.size} rows are not normalized")
    
    # Check 3: Sparsity
    zero_count = np.sum(bmdp_M.p_n_M == 0.0)
    total_entries = cardinality * cardinality * n_u
    sparsity = zero_count / total_entries * 100
    
    print(f"\nSparsity:")
    print(f"  Zero entries: {zero_count:,} / {total_entries:,} ({sparsity:.2f}%)")
    print(f"  Non-zero entries: {total_entries - zero_count:,} ({100 - sparsity:.2f}%)")
    
    # Check 4: Sample some transition probabilities
    print(f"\nSample transitions:")
    print(f"  p_n^{(M)}(π_0 → π_0 | u_0): {bmdp_M.p_n_M[0, 0, 0]:.6e}")
    print(f"  p_n^{(M)}(π_0 → π_1 | u_0): {bmdp_M.p_n_M[0, 1, 0]:.6e}")
    if cardinality > 2:
        print(f"  p_n^{(M)}(π_1 → π_0 | u_0): {bmdp_M.p_n_M[1, 0, 0]:.6e}")
        print(f"  p_n^{(M)}(π_1 → π_1 | u_0): {bmdp_M.p_n_M[1, 1, 0]:.6e}")

print()
print("="*70)
print("Verification complete!")
print("="*70)


In [ ]:
# Show statistics per action
print("="*70)
print("Statistics per action...")
print("="*70)
print()

# Check if sparse format
is_sparse = isinstance(bmdp_M.p_n_M, list)

for k in range(min(5, n_u)):  # Show first 5 actions
    u = bmdp_M.AQ.U[k]
    
    if is_sparse:
        # Sparse format
        p_n_M_u = bmdp_M.p_n_M[k]  # Sparse matrix for action k
        non_zero_count = p_n_M_u.nnz
        
        if non_zero_count > 0:
            data = p_n_M_u.data
            mean_transition = float(np.mean(data))
            max_transition = float(np.max(data))
        else:
            mean_transition = 0.0
            max_transition = 0.0
        
        sparsity = (1.0 - non_zero_count / (cardinality * cardinality)) * 100
    else:
        # Dense format
        p_n_M_u = bmdp_M.p_n_M[:, :, k]  # Transition matrix for action k
        non_zero_count = np.sum(p_n_M_u > 0)
        mean_transition = float(np.mean(p_n_M_u[p_n_M_u > 0])) if non_zero_count > 0 else 0.0
        max_transition = float(np.max(p_n_M_u))
        sparsity = (1.0 - non_zero_count / (cardinality * cardinality)) * 100
    
    print(f"Action {k}: u = [{u[0]:.4f}, {u[1]:.4f}]")
    print(f"  Non-zero transitions: {non_zero_count:,} / {cardinality * cardinality:,}")
    print(f"  Sparsity: {sparsity:.2f}%")
    if non_zero_count > 0:
        print(f"  Mean non-zero transition prob: {mean_transition:.6e}")
    print(f"  Max transition prob: {max_transition:.6e}")
    print()

if n_u > 5:
    print(f"... (showing first 5 of {n_u} actions)")
    print()

print("="*70)
print("p_n^{(M)} generation and verification complete!")
print("="*70)
print()
print("Next steps:")
print("  1. Use this BeliefMDP_n_M instance in value iteration")
print("  2. The cached p_n^{(M)} will be automatically loaded in future runs")
print("  3. To test with finer quantization, increase n and/or M")
